# **Domande di analisi**

In [11]:
import pandas as pd
import numpy as np

In [12]:
pd.set_option('display.max_columns', None)

In [13]:
# dati puliti
distribution_center_clean=pd.read_csv("..\\data\\clean_data\\distribution_center_clean.csv")
events_clean=pd.read_csv("..\\data\\clean_data\\events_clean.csv")
inventory_items_clean=pd.read_csv("..\\data\\clean_data\\inventory_items_clean.csv")
order_items_clean=pd.read_csv("..\\data\\clean_data\\order_items_clean.csv")
orders_clean=pd.read_csv("..\\data\\clean_data\\orders_clean.csv")
products_clean=pd.read_csv("..\\data\\clean_data\\products_clean.csv")
users_clean=pd.read_csv("..\\data\\clean_data\\users_clean.csv")

### Website Activity (events)

In [14]:
events_clean.sample(5)

,session_id,user_id,finish,event_type,traffic_source,state,year,month,country
122423,5737b56a-ebe1-430a-aff6-0c290f70b288,NaN,3,cart,Email,Amazonas,2020,5,Colombia
385791,b42afa59-b739-486e-9c0f-3ede0dff70e9,NaN,3,cart,YouTube,Guangdong,2024,12,China
98971,e90ebeab-ce65-4eef-a6fa-8444dfff8d6b,31082.0,7,purchase,Email,Guangdong,2023,6,China
274187,c0b04e28-96c4-4e3a-83fe-c01684420025,NaN,3,cart,Email,Seoul,2023,6,South Korea
599720,4103855c-0dfe-4f26-aada-0ebdf0797880,NaN,3,cart,Email,Île-de-France,2025,2,France


In [15]:
# 1) Quali sono gli eventi finali più frequenti?

In [16]:
events_clean["event_type"].value_counts()
# L'evento finale più frequente è la visualizzazione del prodotto. 

event_type
product     252391
purchase    182539
cart        126263
cancel      126234
Name: count, dtype: int64

In [17]:
# 2) Qual è il tasso di conversione del sito? (purchase/tutte le sessioni)

In [18]:
total_sessions = len(events_clean)
purchase_sessions = (events_clean['event_type'] == 'purchase').sum() 

conversion_rate = purchase_sessions / total_sessions 
conversion_rate_pct = conversion_rate * 100
print(f"Tasso di conversione: {conversion_rate_pct:.2f}%")

Tasso di conversione: 26.55%


In [19]:
# vediamolo per paese
conversion_by_country = events_clean.groupby('country')['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_country=conversion_by_country.sort_values(ascending=False)
conversion_by_country
# Austria ha nettamente la conversione più alta

country
Austria           0.333333
Germany           0.277584
France            0.270565
United Kingdom    0.269328
Japan             0.268468
Brasil            0.268076
Spain             0.267834
Australia         0.266903
China             0.264209
United States     0.263411
Colombia          0.262182
South Korea       0.259353
Belgium           0.251983
Poland            0.246785
Unknown           0.238095
Name: event_type, dtype: float64

In [ ]:
events_clean["country"].value_counts()
# Da sottolineare che però l'Austria porta solo 21 sessioni

country
China             231029
United States     153004
Brasil             99304
South Korea        36113
France             32044
United Kingdom     31289
Spain              30224
Germany            28622
Japan              16691
Australia          14923
Belgium             8445
Colombia            4043
Poland              1633
Unknown               42
Austria               21
Name: count, dtype: int64

In [21]:
# è cambiato da anno in anno:
conversion_by_year = events_clean.groupby('year',as_index=False)['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_year=conversion_by_year.sort_values(by='year', ascending=True)
conversion_by_year

,year,event_type
0,2019,0.025161
1,2020,0.078263
2,2021,0.134100
3,2022,0.193642
4,2023,0.260278
5,2024,0.344622
6,2025,0.474276
7,2026,0.688533


In [22]:
"""
KPI molto interessante da utilizzare nella dashboard
Misura con DAX: 
Conversion Rate = 
DIVIDE(
    CALCULATE(COUNTROWS(events), events[event_type] = "purchase"),
    COUNTROWS(events)
)
"""

'\nKPI molto interessante da utilizzare nella dashboard\nMisura con DAX: \nConversion Rate = \nDIVIDE(\n    CALCULATE(COUNTROWS(events), events[event_type] = "purchase"),\n    COUNTROWS(events)\n)\n'

In [23]:
# 3) Quali sorgenti portano più sessioni? Quali più acquisti? Quali hanno il miglior conversion rate?

In [24]:
events_clean["traffic_source"].value_counts()

traffic_source
Email       309632
Adwords     206133
YouTube      68795
Facebook     68475
Organic      34392
Name: count, dtype: int64

In [25]:
# vediamo il conversion rate per traffic_source
conversion_by_traffic_source = events_clean.groupby('traffic_source')['event_type'].apply(lambda x: (x == 'purchase').mean())
conversion_by_traffic_source=conversion_by_traffic_source.sort_values(ascending=False)
conversion_by_traffic_source
# tutte molto allineate

traffic_source
Email       0.266248
Facebook    0.266214
YouTube     0.265310
Adwords     0.265018
Organic     0.261398
Name: event_type, dtype: float64

In [26]:
# 4) Quali combinazioni sono più interessanti?

In [27]:
counts = events_clean.groupby(['traffic_source', 'event_type']).size()
percentages = counts.groupby(level=0).apply(lambda x: x / x.sum())
percentages
# Niente di rilevante. Tutto molto simile

traffic_source  traffic_source  event_type
Adwords         Adwords         cancel        0.183648
                                cart          0.182668
                                product       0.368665
                                purchase      0.265018
Email           Email           cancel        0.182985
                                cart          0.184422
                                product       0.366345
                                purchase      0.266248
Facebook        Facebook        cancel        0.184432
                                cart          0.182037
                                product       0.367317
                                purchase      0.266214
Organic         Organic         cancel        0.187834
                                cart          0.185247
                                product       0.365521
                                purchase      0.261398
YouTube         YouTube         cancel        0.183603
                      

In [28]:
# Un'analisi interessante da fare in Power BI potrebbe essere: osservare il traffic source più utilizzato per il paese selezionato.
# esempio, supponiamo di scegliere il Brasile
brasil=events_clean[events_clean["country"]=="Brasil"]
brasil["traffic_source"].value_counts()

traffic_source
Email       44675
Adwords     29862
Facebook     9968
YouTube      9813
Organic      4986
Name: count, dtype: int64

In [29]:
# Organic indica gli utenti che arrivano sul sito senza che ci sia stata una campagna di marketing pagata o un'azione specifica.
# Adwords  si riferisce al traffico proveniente dalle campagne pubblicitarie a pagamento di Google

### Users

In [30]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
53807,36987,Jeremiah,Rose,31,M,Liaoning,Wuzhong,China,Organic,Adult
68536,84102,Carrie,Lopez,50,F,Oregon,Bend,United States,Search,Middle-aged Adult
67788,91486,Joseph,Robinson,26,M,Ohio,Chillicothe,United States,Search,Adult
78135,42909,Jo,Aguilar,52,F,Seoul,Seoul,South Korea,Search,Middle-aged Adult
73105,80156,Mary,Farmer,51,F,Queensland,Monsildale,Australia,Search,Middle-aged Adult


In [31]:
# 1) Distribuzione demografica

In [32]:
# - età media
users_clean["age"].mean()

np.float64(40.9496)

In [33]:
users_clean.groupby("country")["age"].mean().sort_values(ascending=False)
# l'età media è molto simile in tutti i country

country
Poland            42.301370
France            41.225223
Belgium           41.135783
Germany           41.119952
Japan             41.065491
United States     41.004052
Austria           41.000000
China             40.946580
Brasil            40.900542
United Kingdom    40.867973
Australia         40.863974
South Korea       40.696558
Spain             40.609529
Colombia          40.285714
Name: age, dtype: float64

In [34]:
# - geografia
users_clean["country"].value_counts()
# La China ha nettamente più utenti.

country
China             34051
United States     22456
Brasil            14579
South Korea        5230
France             4813
United Kingdom     4696
Germany            4185
Spain              3967
Japan              2382
Australia          2154
Belgium            1252
Poland              219
Colombia             14
Austria               2
Name: count, dtype: int64

In [35]:
# -genere
users_clean['gender'].value_counts(normalize=True) * 100

gender
M    50.043
F    49.957
Name: proportion, dtype: float64

In [36]:
gender_by_country = (users_clean.groupby('country')['gender'].value_counts(normalize=True).rename('percentage').reset_index())
gender_by_country
# tutti intorno al 50%, differenza più grande in Colombia

,country,gender,percentage
0,Australia,M,0.517642
1,Australia,F,0.482358
2,Austria,F,0.500000
3,Austria,M,0.500000
4,Belgium,M,0.514377
5,Belgium,F,0.485623
6,Brasil,M,0.505796
7,Brasil,F,0.494204
8,China,F,0.503832
9,China,M,0.496168


### Logistica: inventory_items e distribution_center

In [37]:
inventory_items_clean.sample(5)

,id,product_id,product_distribution_center_id,in_stock,month_year_creation,days_in_stock
428147,270580,28017,1,1,Nov 2024,465
275218,39633,6331,7,1,Oct 2024,506
477248,379307,25990,5,1,Mar 2024,706
233513,405744,21659,7,1,Mar 2025,335
262772,321004,15880,2,0,Aug 2024,57


In [38]:
distribution_center_clean

,id,name
0,3,Houston TX
1,7,Philadelphia PA
2,5,New Orleans LA
3,4,Los Angeles CA
4,8,Mobile AL
5,1,Memphis TN
6,2,Chicago IL
7,6,Port Authority of New York/New Jersey NY/NJ
8,10,Savannah GA
9,9,Charleston SC


In [39]:
# 1) Stock per distribution center

# Calcolo articoli in stock (=1)
in_stock = (
    inventory_items_clean
    .assign(in_stock_flag = inventory_items_clean["in_stock"] == 1)
    .groupby("product_distribution_center_id")["in_stock_flag"]
    .sum()
    .reset_index()
    .rename(columns={"in_stock_flag": "items_in_stock"})
)

# Calcolo articoli esauriti (=0)
out_of_stock = (
    inventory_items_clean
    .assign(out_stock_flag = inventory_items_clean["in_stock"] == 0)
    .groupby("product_distribution_center_id")["out_stock_flag"]
    .sum()
    .reset_index()
    .rename(columns={"out_stock_flag": "items_out_of_stock"})
)

# Merge dei due risultati
dist_stock = (
    in_stock
    .merge(out_of_stock, on="product_distribution_center_id")
    .merge(distribution_center_clean, left_on="product_distribution_center_id", right_on="id")
)

# Selezione colonne finali
dist_stock = dist_stock[["id", "name", "items_in_stock", "items_out_of_stock"]]

dist_stock


,id,name,items_in_stock,items_out_of_stock
0,1,Memphis TN,40999,24005
1,2,Chicago IL,40636,23977
2,3,Houston TX,38813,22848
3,4,Los Angeles CA,29735,17477
4,5,New Orleans LA,22076,12952
5,6,Port Authority of New York/New Jersey NY/NJ,27699,16211
6,7,Philadelphia PA,28855,16941
7,8,Mobile AL,31043,18304
8,9,Charleston SC,28476,16831
9,10,Savannah GA,20087,11858


In [40]:
# 2) media dei giorni che un prodotto rimane in stock prima di essere venduto, a quelli non venduti è stato calcolato come la differenza tra la data di oggi (19/02/2026) e la data di creazione
inventory_items_clean.groupby("product_distribution_center_id", as_index=False).agg({"days_in_stock": "mean"}).sort_values("days_in_stock", ascending=False)

,product_distribution_center_id,days_in_stock
2,3,720.287670
0,1,719.731740
6,7,719.282099
8,9,718.777496
9,10,717.820066
5,6,716.383990
7,8,716.254808
3,4,715.231509
1,2,714.541609
4,5,712.507765


In [41]:
# 3) Quali categorie di prodotti sono più presenti in ogni centro?

In [42]:
inv_prod_cat=pd.merge(left=inventory_items_clean, right=products_clean, left_on="product_id", right_on="id", how="left")
inv_prod_cat=inv_prod_cat[["id_x","product_id","product_distribution_center_id","in_stock","category"]]
inv_prod_cat

,id_x,product_id,product_distribution_center_id,in_stock,category
0,174571,13844,7,0,Accessories
1,174572,13844,7,1,Accessories
2,216419,13844,7,0,Accessories
3,216420,13844,7,1,Accessories
4,216421,13844,7,1,Accessories
...,...,...,...,...,...
489818,332010,25590,3,1,Underwear
489819,467577,25590,3,0,Underwear
489820,467578,25590,3,1,Underwear
489821,467579,25590,3,1,Underwear


In [43]:
best_cat_dist = inv_prod_cat.groupby(["product_distribution_center_id", "category"], as_index=False)["in_stock"].sum()
best_cat_dist["rank"] = best_cat_dist.groupby("product_distribution_center_id")["in_stock"].rank(method="dense", ascending=False)
best_cat_dist = best_cat_dist.sort_values(["product_distribution_center_id", "rank"])
best_cat_dist = best_cat_dist[best_cat_dist["rank"] == 1]
best_cat_dist

,product_distribution_center_id,category,in_stock,rank
24,1,Tops & Tees,3121,1.0
32,2,Intimates,4823,1.0
58,3,Intimates,3676,1.0
101,4,Swim,4029,1.0
111,5,Jeans,2009,1.0
153,6,Tops & Tees,2246,1.0
161,7,Jeans,3583,1.0
187,8,Jeans,3761,1.0
210,9,Fashion Hoodies & Sweatshirts,2886,1.0
255,10,Underwear,3050,1.0


### Analisi sui prodotti (products, order_items e users)

In [44]:
products_clean.sample(5)

,id,cost,category,brand,retail_price,department,distribution_center_id,ideal_profit
204,306,23.73,Tops & Tees,ASICS,45.99,Women,1,22.26
11646,15563,17.81,Plus,H2W,38.97,Women,4,21.16
946,4233,40.02,Jeans,Seven7,69.00,Women,1,28.98
1603,23255,20.88,Shorts,Rip Curl,43.95,Men,1,23.07
1820,18609,44.65,Active,Sweatsedo,99.00,Men,1,54.35


In [45]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
61578,65864,James,Rhodes,24,M,New South Wales,Kyeamba,Australia,Search,Young
42088,79713,Joshua,Wright,22,M,Heilongjiang,Wenzhou,China,Organic,Young
42249,4914,Traci,Bryant,51,F,Heilongjiang,Zhoukou,China,Search,Middle-aged Adult
48163,68801,Rachel,Roach,42,F,Indiana,Merrillville,United States,Search,Adult
77749,56192,Regina,Richardson,38,F,Seoul,Seoul,South Korea,Search,Adult


In [46]:
order_items_clean.sample(5)

,id,order_id,user_id,product_id,inventory_item_id,status,sale_price,cost,profit_per_product
176092,116323,80306,64260,24408,313945,Complete,200.00,96.00,104.00
141997,48847,33919,27341,11869,131692,Shipped,78.00,40.17,37.83
40774,167551,115500,92463,29029,452391,Processing,22.39,7.95,14.44
23302,95416,66000,52815,7994,257460,Complete,15.37,5.55,9.82
90150,174617,120396,96349,15293,471442,Complete,39.99,20.51,19.48


In [47]:
# Creiamo una tabella unica che ci aiuterà nell'analisi 
all=pd.merge(left=order_items_clean, right=products_clean, how='left', left_on='product_id', right_on='id')
all=all[["id_x","order_id","user_id","product_id","status","sale_price","cost_x","profit_per_product","category","brand","department"]]
all.rename(columns={"id_x": "id", "cost_x": "cost"}, inplace=True)
all.sample(5)

,id,order_id,user_id,product_id,status,sale_price,cost,profit_per_product,category,brand,department
99224,150914,104125,83313,23155,Cancelled,44.99,24.02,20.97,Shorts,Affliction,Men
140686,152876,105511,84394,8058,Shipped,75.00,43.65,31.35,Clothing Sets,Taraluna,Women
118331,17053,11831,9493,27994,Processing,55.00,33.66,21.34,Swim,Hurley,Men
19028,6516,4513,3587,25709,Shipped,14.00,5.96,8.04,Underwear,Jockey,Men
84181,99817,69017,55215,1124,Complete,37.95,16.43,21.52,Sweaters,Romeo & Juliet Couture,Women


In [48]:
all2=pd.merge(all, users_clean, left_on='user_id', right_on='id', how='left')
all2=all2[["id_x", "order_id", "user_id", "product_id", "status", "sale_price", "cost", "profit_per_product", "category", "brand", "department", "gender", "country", "age_group"]]
all2.rename(columns={"id_x": "id"}, inplace=True)
all2.sample(5)

,id,order_id,user_id,product_id,status,sale_price,cost,profit_per_product,category,brand,department,gender,country,age_group
144286,147371,101699,81359,3541,Complete,79.99,39.76,40.23,Dresses,Folter,Women,F,South Korea,Adult
87535,87899,60820,48649,6615,Processing,39.50,19.28,20.22,Shorts,Calvin Klein Jeans,Women,F,United States,Middle-aged Adult
156465,78299,54209,43361,4600,Shipped,100.99,56.25,44.74,Jeans,DL1961,Women,F,South Korea,Adult
155360,84996,58839,47018,3115,Cancelled,99.99,46.10,53.89,Dresses,Jessica Howard,Women,F,United States,Young
37265,1889,1317,1062,17030,Processing,20.85,10.95,9.90,Tops & Tees,WinnieFashion,Men,M,Brasil,Adult


In [49]:
# 1) Analizziamo le categorie

In [50]:
# teniamo solo gli ordini in processo, spediti, completati e non restituiti o cancellati
all_no_ret_canc = all2[(all2["status"] != "Returned") & (all2["status"] != "Cancelled")]

In [51]:
# categorie più vendute
cat_most_sal=all_no_ret_canc.groupby("category",as_index=False)["product_id"].size().sort_values(by="size",ascending=False)
cat_most_sal

,category,size
6,Intimates,10082
7,Jeans,9714
24,Tops & Tees,9064
5,Fashion Hoodies & Sweatshirts,8933
15,Shorts,8447
22,Sweaters,8433
23,Swim,8415
17,Sleep & Lounge,8395
0,Accessories,7412
1,Active,6792


In [52]:
# saranno le stesse 3 ad avere il miglior profitto?
all_no_ret_canc.groupby("category", as_index=False)["profit_per_product"].sum().sort_values("profit_per_product", ascending=False).head(3)

,category,profit_per_product
11,Outerwear & Coats,547393.18
7,Jeans,441072.89
22,Sweaters,330524.90


In [53]:
"""
Potremmo osservare tantissime cose. La mia idea è che in Power BI andiamo a selezionare un country 
e per esso visualizziamo le categorie più profittevoli, le più vendute, con una suddivisione anche per gruppo d'età.
"""

"\nPotremmo osservare tantissime cose. La mia idea è che in Power BI andiamo a selezionare un country \ne per esso visualizziamo le categorie più profittevoli, le più vendute, con una suddivisione anche per gruppo d'età.\n"

In [54]:
# 2) focus sui resi

In [55]:
all_ret=all2[(all2["status"] == "Returned")]

In [56]:
cat_most_ret=all_ret.groupby(["category"],as_index=False)["product_id"].size().sort_values(by="size", ascending=False)
cat_most_ret

,category,size
6,Intimates,1317
7,Jeans,1282
24,Tops & Tees,1247
5,Fashion Hoodies & Sweatshirts,1183
15,Shorts,1145
17,Sleep & Lounge,1141
23,Swim,1131
22,Sweaters,1114
0,Accessories,978
1,Active,938


In [57]:
# Calcoliamo la percentuale di prodotti resi rispetto al totale

In [58]:
perc_ret=pd.merge(cat_most_sal, cat_most_ret, on='category', how='inner')
perc_ret.rename(columns={'size_x':'saled', 'size_y':'returned'}, inplace=True)
perc_ret["total"] = perc_ret["saled"] + perc_ret["returned"]
perc_ret["perc_returned"] = round((perc_ret["returned"] / perc_ret["total"])*100,2)
perc_ret=perc_ret[["category", "perc_returned"]]
perc_ret=perc_ret.sort_values(by='perc_returned', ascending=False)
perc_ret
# la percentuale di reso può essere una misura interessante da utilizzare in Power BI

,category,perc_returned
25,Clothing Sets,13.56
21,Blazers & Jackets,13.12
23,Suits,12.78
17,Plus,12.69
18,Socks & Hosiery,12.52
24,Jumpsuits & Rompers,12.42
20,Leggings,12.36
9,Active,12.13
10,Outerwear & Coats,12.11
2,Tops & Tees,12.09


In [59]:
""" Misura percentuale resi in DAX:
Returned Items = 
CALCULATE(
    COUNTROWS(order_items),
    order_items[status] = "Returned"
)

Sold Items =
CALCULATE(
    COUNTROWS(order_items),
    order_items[status] <> "Cancelled"
)

Return Rate = 
DIVIDE([Returned Items], [Sold Items])
"""

' Misura percentuale resi in DAX:\nReturned Items = \nCALCULATE(\n    COUNTROWS(order_items),\n    order_items[status] = "Returned"\n)\n\nSold Items =\nCALCULATE(\n    COUNTROWS(order_items),\n    order_items[status] <> "Cancelled"\n)\n\nReturn Rate = \nDIVIDE([Returned Items], [Sold Items])\n'

### Analisi sugli ordini (orders, users)

In [60]:
orders_clean.sample(5)

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,year,month,sale_price,cost,profit
36783,118061,94499,Processing,1,Dec 2025,NaN,NaN,2025,12,28.00,14.42,13.58
81810,79967,64009,Complete,1,Jan 2025,1.0,3.0,2025,1,179.00,64.26,114.74
101705,34194,27565,Returned,1,Jun 2024,0.0,2.0,2024,6,119.99,49.44,0.00
73849,16149,12948,Complete,1,Sep 2025,1.0,2.0,2025,9,129.00,65.40,63.60
123918,118149,94579,Shipped,2,Oct 2025,1.0,NaN,2025,10,62.06,31.19,30.87


In [61]:
users_clean.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
3373,14788,Robert,Moore,12,M,Arizona,Chandler,United States,Search,Young
99788,11387,Stacey,Ferguson,69,F,Île-de-France,Tigery,France,Search,Senior
85662,54258,Katelyn,Austin,24,F,São Paulo,Mirassol,Brasil,Display,Young
89328,47717,Ralph,Jones,39,M,Texas,Arlington,United States,Search,Adult
91700,84873,Jason,Schroeder,60,M,Tokyo,Musashino,Japan,Search,Middle-aged Adult


In [62]:
# Creare tabella unica per facilitare le analisi
ord_use=pd.merge(orders_clean, users_clean, left_on='user_id', right_on='id', how='left')
ord_use=ord_use[["order_id", "user_id","status","num_of_item","month_year_creation","days_for_shipping","days_for_delivery",
"year","month","sale_price","cost","profit","age","gender","country","age_group"]]
ord_use.sample(5)

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,year,month,sale_price,cost,profit,age,gender,country,age_group
30656,57691,46077,Processing,1,Dec 2025,NaN,NaN,2025,12,34.00,17.48,16.52,51,F,United States,Middle-aged Adult
96388,88912,71143,Processing,1,Jun 2025,NaN,NaN,2025,6,5.95,3.87,2.08,17,M,Germany,Young
11109,13671,11014,Complete,1,Oct 2024,0.0,0.0,2024,10,18.95,9.11,9.84,45,F,China,Middle-aged Adult
55273,76410,61137,Shipped,4,Apr 2025,1.0,NaN,2025,4,144.48,72.81,71.67,13,F,China,Young
90998,34476,27789,Processing,1,Feb 2026,NaN,NaN,2026,2,25.00,10.43,14.57,21,M,United States,Young


In [63]:
# 1) paesi con più ordini e con più profitto

In [64]:
ord_use.groupby("country", as_index=False)["order_id"].size().sort_values(by="size", ascending=False)

,country,size
4,China,42704
13,United States,28022
3,Brasil,18046
10,South Korea,6461
6,France,6014
12,United Kingdom,5874
7,Germany,5360
11,Spain,4922
8,Japan,3076
0,Australia,2669


In [65]:
ord_use.groupby("country", as_index=False)["profit"].sum().sort_values(by="profit", ascending=False)

,country,profit
4,China,1442929.35
13,United States,946597.09
3,Brasil,592450.48
10,South Korea,219388.09
12,United Kingdom,203871.85
6,France,199724.04
7,Germany,180809.12
11,Spain,164628.17
8,Japan,99856.11
0,Australia,89167.92


In [66]:
ord_use.groupby("country", as_index=False)["profit"].mean().sort_values(by="profit", ascending=False)

,country,profit
12,United Kingdom,34.707499
9,Poland,34.176245
10,South Korea,33.955748
4,China,33.789091
13,United States,33.780497
7,Germany,33.733045
2,Belgium,33.489922
11,Spain,33.447414
0,Australia,33.408737
6,France,33.209850


In [67]:
# Colombia e Austria sono le peggiori per numero di ordini, profitto totale e media del profitto per ogni ordine

In [68]:
# 2) Come sta andando nel corso degli anni?

In [69]:
ord_use.groupby("year", as_index=False)["profit"].sum().sort_values(by="year", ascending=True)
# Molto in crescita, il 2026 ha solo i primi due mesi.
# Sarà interessante vederlo su PowerBI, confrontando i vari paesi.

,year,profit
0,2019,41849.21
1,2020,134726.74
2,2021,250746.86
3,2022,385549.14
4,2023,592632.63
5,2024,860071.09
6,2025,1473116.44
7,2026,462389.02


In [70]:
ord_use["month_year_creation"].value_counts()

month_year_creation
Feb 2026    7128
Jan 2026    6661
Dec 2025    5437
Nov 2025    4734
Oct 2025    4393
            ... 
May 2019      83
Apr 2019      71
Mar 2019      51
Feb 2019      22
Jan 2019      16
Name: count, Length: 86, dtype: int64

In [71]:
# cambia il modo di acquistare in base all'età?

In [72]:
ord_use.groupby("age_group", as_index=False)["sale_price"].mean().sort_values(by="sale_price", ascending=False)
# Anche qui non sembra incidere più di tanto a livello generale, magari in qualche paese c'è una differenza più marcata. 

,age_group,sale_price
2,Senior,87.003372
1,Middle-aged Adult,86.503713
0,Adult,86.454370
3,Young,86.197559


In [73]:
# 3) stagionalità
# Supponiamo di concentraric sul brasile: in quali mesi spendono di più? in quali fanno più acquisti?

In [74]:
ord_use[ord_use['country']=='Brasil'].groupby("month", as_index=False)["sale_price"].mean().sort_values(by="sale_price", ascending=False)

,month,sale_price
3,4,91.156303
11,12,88.378047
2,3,87.420597
0,1,86.980224
9,10,86.381895
6,7,86.267607
7,8,85.965105
10,11,84.943306
4,5,83.906693
8,9,83.617734


In [75]:
ord_use[ord_use['country']=='Brasil'].groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)
# Nel periodo tra dicembre e febbraio i brasiliani tendono a fare più acquisti.

,month,size
0,1,2055
1,2,1987
2,3,1105
3,4,1147
4,5,1261
5,6,1222
6,7,1425
7,8,1379
8,9,1390
9,10,1562


In [76]:
# negli Stati Uniti è la stessa cosa? sembra di si
ord_use[ord_use['country']=='United States'].groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)

,month,size
0,1,3113
1,2,3063
2,3,1883
3,4,1816
4,5,2001
5,6,1901
6,7,2104
7,8,2190
8,9,2229
9,10,2382


In [77]:
# è una tendenza generale del sito?

In [78]:
ord_use.groupby("month", as_index=False)["order_id"].size().sort_values(by="month", ascending=True)
# Sembra proprio di sì. I periodi invernali sono quelli che portano a più acquisti.

,month,size
0,1,13950
1,2,13938
2,3,7858
3,4,8121
4,5,8693
5,6,8598
6,7,9293
7,8,9803
8,9,9953
9,10,10863


In [79]:
ord_use.groupby("user_id", as_index=False)["order_id"].size().sort_values(by="size", ascending=False).head(10)
# Il massimo numero di ordini per un singolo utente è 4. Sono in tanti con 4 ordini. Vediamo se otteniamo informazioni migliori tenendo conto della spesa.

,user_id,size
29798,37288,4
74115,92684,4
42137,52676,4
26420,33093,4
74105,92671,4
18742,23542,4
26427,33101,4
14354,18023,4
9987,12534,4
48567,60740,4


In [80]:
# 10 clienti che hanno speso di più
top10_users=ord_use.groupby("user_id", as_index=False)["sale_price"].sum().sort_values(by="sale_price", ascending=False).head(10)

In [81]:
top10_users_merged = top10_users.merge(users_clean, left_on="user_id", right_on="id", how='inner')
top10_users_merged

,user_id,sale_price,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
0,58819,1817.70,58819,Leonard,Smith,45,M,Guangdong,Wuhan,China,Search,Middle-aged Adult
1,56589,1707.97,56589,Richard,Smith,16,M,Cataluña,Terrassa,Spain,Search,Young
2,34715,1663.45,34715,Joseph,Davis,26,M,Beijing,Weihai,China,Organic,Adult
3,32484,1586.62,32484,Lynn,Mathews,42,F,Shanghai,Qinhuangdao,China,Search,Adult
4,12322,1558.28,12322,Anthony,King,67,M,Berlin,Berlin,Germany,Email,Senior
5,71039,1497.87,71039,Carly,Villa,62,F,Henan,Fushun,China,Search,Middle-aged Adult
6,26088,1491.98,26088,Kerri,Freeman,12,F,Sichuan,Wenzhou,China,Organic,Young
7,38791,1451.62,38791,Frances,Lucas,13,F,Shanghai,Nanchong,China,Facebook,Young
8,18010,1392.18,18010,Diane,Keller,51,F,Oregon,Newberg,United States,Organic,Middle-aged Adult
9,83570,1391.49,83570,Jessica,Li,70,F,Minas Gerais,Uberlândia,Brasil,Search,Senior


In [83]:
top10_users_merged=top10_users_merged[['country', 'age', 'gender', 'sale_price']]
top10_users_merged.columns =['country', 'age', 'gender', 'money_spend']

In [84]:
top10_users_merged
# 6 cinesi, 2 giovanissime

,country,age,gender,money_spend
0,China,45,M,1817.70
1,Spain,16,M,1707.97
2,China,26,M,1663.45
3,China,42,F,1586.62
4,Germany,67,M,1558.28
5,China,62,F,1497.87
6,China,12,F,1491.98
7,China,13,F,1451.62
8,United States,51,F,1392.18
9,Brasil,70,F,1391.49
